# Data Cleaning — Parkinson's Disease (Longitudinal + Single-session)
Load the raw parquet tables saved from `explore_longitudinal.ipynb`, inspect, clean, and prepare for modelling.

In [66]:
import numpy as np
import pandas as pd
from pathlib import Path

# ── Load the two Parkinson's tables ──────────────────────────────
DATA_DIR = Path("/home/srl/B2AI/LLM/data")

pk_train = pd.read_parquet(DATA_DIR / "raw_data_non_longitudinal_parkinson.parquet")

## 1. Inspect raw data
Show column types, NaN percentages, and a few sample rows for each table.

In [67]:
def inspect_table(df, name):
    """Print detailed column-level summary for a DataFrame."""
    print(f"\n{'═' * 80}")
    print(f"  {name}   shape: {df.shape}")
    print(f"{'═' * 80}")

    # Columns that contain arrays (not scalar) — skip nunique/sample for these
    ARRAY_COLS = {"mfcc", "mel_spectrogram"}

    rows = len(df)
    report = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        n_na = df[col].isna().sum()
        pct_na = 100 * n_na / rows if rows > 0 else 0

        if col in ARRAY_COLS:
            n_unique = "—"
            sample = f"<array len={len(df[col].dropna().iloc[0]) if n_na < rows else 0}>"
        else:
            n_unique = df[col].nunique()
            non_null = df[col].dropna()
            sample = str(non_null.iloc[0])[:40] if len(non_null) > 0 else "—"

        report.append({
            "column": col, "dtype": dtype,
            "NaN": n_na, "NaN%": round(pct_na, 1),
            "unique": n_unique, "sample": sample,
        })

    report_df = pd.DataFrame(report)
    display(report_df)

    # Highlight columns with >60% NaN
    high_nan = report_df[report_df["NaN%"] > 60]
    if len(high_nan) > 0:
        print(f"\n  ⚠  {len(high_nan)} columns with > 60% NaN:")
        for _, r in high_nan.iterrows():
            print(f"    • {r['column']:55s}  {r['NaN%']:5.1f}%")
    else:
        print(f"\n  ✔  No columns with > 60% NaN")

    return report_df

report = inspect_table(pk_train, "pk_train (non-longitudinal)")


════════════════════════════════════════════════════════════════════════════════
  pk_train (non-longitudinal)   shape: (1876, 135)
════════════════════════════════════════════════════════════════════════════════


,column,dtype,NaN,NaN%,unique,sample
0,participant_id,str,0,0.0,56,110043
1,session_id,str,0,0.0,56,2cd1e0d6
2,task_name,str,0,0.0,120,cinderella-story
3,mfcc,object,0,0.0,—,<array len=60>
4,n_frames_mfcc,int64,0,0.0,985,3278
...,...,...,...,...,...,...
130,eligible_studies___4,str,1732,92.3,1,1.0
131,eligible_studies___age_10_plus,str,1876,100.0,0,—
132,eligible_studies___age_2_4,str,1876,100.0,0,—
133,eligible_studies___age_4_6,str,1876,100.0,0,—



  ⚠  61 columns with > 60% NaN:
    • colorblind                                                91.0%
    • live_by_yourself                                          98.2%
    • primary_transportation                                    85.4%
    • roommate                                                 100.0%
    • sex_assigned_at_birth                                     91.0%
    • tone_deaf                                                 91.0%
    • transportation                                           100.0%
    • demographics_completed_by___2                             92.3%
    • demographics_completed_by___3                             98.0%
    • employ_status___1                                         98.1%
    • employ_status___10                                       100.0%
    • employ_status___11                                       100.0%
    • employ_status___12                                       100.0%
    • employ_status___13                                 

## 2. Drop columns with > 60% NaN
Remove columns that are mostly empty — they carry little information and add noise.

In [68]:
NAN_THRESHOLD = 0.60  # drop columns with more than 60% NaN

def drop_high_nan(df, name, threshold=NAN_THRESHOLD):
    """Drop columns with NaN% > threshold. Returns cleaned DataFrame."""
    n_rows = len(df)
    nan_pct = df.isna().sum() / n_rows
    drop_cols = nan_pct[nan_pct > threshold].index.tolist()

    print(f"{'═' * 70}")
    print(f"  {name}")
    print(f"  Before: {df.shape[1]} columns")
    print(f"  Dropping {len(drop_cols)} columns with > {threshold*100:.0f}% NaN:")

    df_clean = df.drop(columns=drop_cols)
    print(f"  After:  {df_clean.shape[1]} columns")
    print(f"{'═' * 70}\n")
    return df_clean

pk_train_c = drop_high_nan(pk_train, "pk_train (non-longitudinal)")

══════════════════════════════════════════════════════════════════════
  pk_train (non-longitudinal)
  Before: 135 columns
  Dropping 61 columns with > 60% NaN:
  After:  74 columns
══════════════════════════════════════════════════════════════════════



## 3. Column mappings (from `demographics.json`)

Five kinds of encoding, derived from the data dictionary:

| Kind | Columns | Rule |
|------|---------|------|
| **Ordinal** | `edu_level`, `household_income_usa`, `housing_status` | Text label → increasing integer. **"Prefer not to answer"** / **"Other"** → `NaN`. |
| **Boolean** | `cognition`, `hearing`, `mobility`, `sex_at_birth`, … | `Yes → 1`, `No → 0`, `Prefer not to answer → NaN`. |
| **Checked** | `self_reported_*` | `Checked → 1`, `Unchecked → 0`. |
| **One-hot** | `country`, `sexual_orientation` | Dummy columns per category. `Prefer not to answer → NaN`. |

In [69]:
# ══════════════════════════════════════════════════════════════════
#  3a. ORDINAL columns  — text → increasing integer
#      "Prefer not to answer" / "Other" are EXCLUDED → become NaN
# ══════════════════════════════════════════════════════════════════

ORDINAL_MAP = {
    # ── Education level (1 = lowest → 10 = highest) ─────────────
    "edu_level": {
        "No formal education":                              1,
        "Some elementary school":                           2,
        "Some secondary or high school education":          3,
        "High School or secondary school degree complete":  4,
        "Some college education":                           5,
        "Associate's or technical degree complete":         6,
        "College or baccalaureate degree complete":         7,
        "Some post-baccalaureate education":                8,
        "Graduate or professional degree complete":         9,
        "Doctoral or post graduate education":             10,
        # "Other" (11 in JSON) → NaN  — excluded from map
        # "Prefer not to answer" (12 in JSON) → NaN  — excluded from map
    },

    # ── Household income  (1 = lowest → 8 = highest) ────────────
    "household_income_usa": {
        "< $15,000":              1,
        "$15,000 to $29,999":     2,
        "$30,000 to $$49,999":    3,   # double-$ typo in source JSON
        "$30,000 to $49,999":     3,   # in case data has corrected form
        "$50,000 to $99,999":     4,
        "$100,000 to $149,999":   5,
        "$150,000 to $199,999":   6,
        "$200,000 to $249,999":   7,
        ">$250,000":              8,
        # "Prefer not to answer" (9 in JSON) → NaN  — excluded from map
    },

    # ── Housing status (1 = most dependent → 3 = most independent) ──
    "housing_status": {
        "Assisted living":                          1,
        "Skilled nursing facility/nursing home":    1,   # same level as assisted
        "Rent home":                                2,
        "Own home":                                 3,
        # Any other / unknown → NaN
    },
}

In [70]:
# ══════════════════════════════════════════════════════════════════
#  3b. BOOLEAN columns  — Yes→1, No→0, "Prefer not to answer"→NaN
#      Kept as a separate list so it's easy to iterate
# ══════════════════════════════════════════════════════════════════

BOOL_COLS = [
    # Demographics — disability / function questions
    "cognition",            # difficulty concentrating / remembering
    "hearing",              # deaf or serious difficulty hearing
    "independent_living",   # difficulty doing errands alone
    "mobility",             # difficulty walking / climbing stairs
    "self_care",            # difficulty dressing / bathing
    "vision",               # blind or serious difficulty seeing
    # Demographics — household composition (JSON: Yes=1, No=0)
    "children",
    "grandparent",
    "parent",
    "other_live_with",
    "spouse_partner_sig_other",
    # Ethnicity (Hispanic or Latino = yes / Not Hispanic or Latino = no)
    "ethnicity",
    # Enrollment flags
    "is_control_participant",
    "ef_any_cognitive_challeges",
    "ef_any_hearing_related_req",
    "ef_any_physical_challenges",
    "ef_any_vision_related_req",
    "sex_at_birth",
]

# Master mapping — covers all text variants found in the data
BOOL_MAP = {
    # Standard yes / no
    "Yes": 1,   "yes": 1,
    "No":  0,   "no":  0,
    # Ethnicity-specific labels
    "Hispanic or Latino":       1,
    "Not Hispanic or Latino":   0,
    # All ambiguous / refusal answers → NaN
    "Prefer not to answer":  np.nan,
    "noAnswer":              np.nan,
    "Not certain":           np.nan,
    "notCertain":            np.nan,
    "Male":                 1,
    "Female":               0,
}

# ══════════════════════════════════════════════════════════════════
#  3c. CHECKED columns  — Checked→1, Unchecked→0
#      (self_reported_* from enrollment)
# ══════════════════════════════════════════════════════════════════

CHECKED_MAP = {"Checked": 1, "Unchecked": 0}

In [71]:
# ══════════════════════════════════════════════════════════════════
#  3d. Apply all mappings + coerce remaining numeric columns
# ══════════════════════════════════════════════════════════════════

# Columns that should be coerced to numeric (already numbers or convertible)
NUMERIC_COLS = [
    "parkinsons_disease__diagnosis_parkinsons_ds",
    "diagnosis_parkinsons_ds",
    "vhi10__vhi_10_calc_score",
    "voice_perception__voice_quality_perception",
    "age",
    "household_count",
]

# Columns to drop (form-fill time / single-value — not useful for modelling)
DROP_COLS = ["demographics_duration", "ef_primary_language", "ef_select_language"]

# ── One-hot encoding targets ────────────────────────────────────
#  country:            USA / Canada  → one-hot (drop_first → 1 binary col)
#  sexual_orientation: Heterosexual / Bisexual  (Prefer not to answer → NaN first)
ONE_HOT_COLS = ["country", "sexual_orientation"]

# Values that should become NaN before one-hot encoding
ONE_HOT_NAN_VALUES = {"Prefer not to answer", "noAnswer"}


def clean_types(df, name):
    """Apply all mappings, coerce types, and drop useless columns."""
    df = df.copy()

    # ── 1. Drop useless columns ──────────────────────────────────
    existing_drop = [c for c in DROP_COLS if c in df.columns]
    if existing_drop:
        df = df.drop(columns=existing_drop)
        print(f"  [{name}] Dropped: {existing_drop}")

    # ── 2. Ordinal mapping ───────────────────────────────────────
    n_ordinal = 0
    for col, mapping in ORDINAL_MAP.items():
        if col not in df.columns:
            continue
        before_na = df[col].isna().sum()
        df[col] = df[col].map(mapping)          # unmapped values → NaN
        after_na = df[col].isna().sum()
        new_na = after_na - before_na
        n_ordinal += 1
        print(f"  [{name}] Ordinal  {col:40s}  ({new_na} unmapped → NaN)")

    # ── 3. Boolean mapping ───────────────────────────────────────
    n_bool = 0
    for col in BOOL_COLS:
        if col not in df.columns:
            continue
        before_na = df[col].isna().sum()
        df[col] = df[col].map(BOOL_MAP)         # unmapped values → NaN
        after_na = df[col].isna().sum()
        new_na = after_na - before_na
        n_bool += 1
        if new_na > 0:
            print(f"  [{name}] Bool     {col:40s}  ({new_na} → NaN)")

    # ── 4. Checked / Unchecked mapping (self_reported_*) ─────────
    checked_cols = [c for c in df.columns if c.startswith("self_reported_")]
    for col in checked_cols:
        df[col] = df[col].map(CHECKED_MAP)

    # ── 5. One-hot encoding (country, sexual_orientation) ────────
    n_onehot = 0
    for col in ONE_HOT_COLS:
        if col not in df.columns:
            continue
        # "Prefer not to answer" → NaN before encoding
        df[col] = df[col].where(~df[col].isin(ONE_HOT_NAN_VALUES))
        dummies = pd.get_dummies(df[col], prefix=col, dummy_na=False)
        dummies = dummies.astype("Int8")         # nullable int to preserve NaN rows
        # Where original was NaN → set all dummies to NaN
        nan_mask = df[col].isna()
        dummies[nan_mask] = pd.NA
        df = df.drop(columns=[col])
        df = pd.concat([df, dummies], axis=1)
        n_onehot += 1
        print(f"  [{name}] One-hot  {col:40s}  → {list(dummies.columns)}")

    # ── 6. Coerce remaining numeric columns (age, scores, etc.) ──
    coerced = []
    for col in NUMERIC_COLS:
        if col in df.columns:
            before_na = df[col].isna().sum()
            df[col] = pd.to_numeric(df[col], errors="coerce")
            after_na = df[col].isna().sum()
            new_na = after_na - before_na
            coerced.append(col)
            if new_na > 0:
                print(f"  [{name}] Numeric  {col}: {new_na} non-numeric → NaN")

    # ── 7. Coerce flag columns (___) to numeric ──────────────────
    flag_cols = [c for c in df.columns if "___" in c]
    for col in flag_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── Summary ──────────────────────────────────────────────────
    print(f"\n  [{name}] SUMMARY:")
    print(f"    Ordinal mapped:   {n_ordinal} columns")
    print(f"    Boolean mapped:   {n_bool} columns")
    print(f"    Checked mapped:   {len(checked_cols)} self_reported_* columns")
    print(f"    One-hot encoded:  {n_onehot} columns")
    print(f"    Numeric coerced:  {len(coerced)} columns")
    print(f"    Flag coerced:     {len(flag_cols)} (___) columns")
    print(f"    Final shape:      {df.shape}")

    return df


pk_train_c = clean_types(pk_train_c, "pk_train_c")

  [pk_train_c] Dropped: ['demographics_duration', 'ef_primary_language', 'ef_select_language']
  [pk_train_c] Ordinal  edu_level                                 (102 unmapped → NaN)
  [pk_train_c] Ordinal  household_income_usa                      (170 unmapped → NaN)
  [pk_train_c] Ordinal  housing_status                            (0 unmapped → NaN)
  [pk_train_c] Bool     cognition                                 (34 → NaN)
  [pk_train_c] Bool     hearing                                   (36 → NaN)
  [pk_train_c] Bool     independent_living                        (34 → NaN)
  [pk_train_c] Bool     ethnicity                                 (35 → NaN)
  [pk_train_c] One-hot  country                                   → ['country_Canada', 'country_USA']
  [pk_train_c] One-hot  sexual_orientation                        → ['sexual_orientation_Bisexual', 'sexual_orientation_Heterosexual (straight)']

  [pk_train_c] SUMMARY:
    Ordinal mapped:   3 columns
    Boolean mapped:   18 columns


In [72]:
list(pk_train_c["task_name"].unique())



['cinderella-story',
 'diadochokinesis-buttercup',
 'diadochokinesis-ka',
 'diadochokinesis-pa',
 'diadochokinesis-pataka',
 'diadochokinesis-ta',
 'glides-high-to-low',
 'glides-low-to-high',
 'loudness',
 'maximum-phonation-time-1',
 'maximum-phonation-time-2',
 'maximum-phonation-time-3',
 'picture-description',
 'productive-vocabulary-1',
 'productive-vocabulary-2',
 'productive-vocabulary-3',
 'productive-vocabulary-4',
 'productive-vocabulary-5',
 'productive-vocabulary-6',
 'prolonged-vowel',
 'rainbow-passage',
 'random-item-generation',
 'respiration-and-cough-breath-1',
 'respiration-and-cough-breath-2',
 'respiration-and-cough-cough-1',
 'respiration-and-cough-cough-2',
 'respiration-and-cough-fivebreaths-1',
 'respiration-and-cough-fivebreaths-2',
 'respiration-and-cough-fivebreaths-3',
 'respiration-and-cough-fivebreaths-4',
 'respiration-and-cough-threequickbreaths-1',
 'respiration-and-cough-threequickbreaths-2',
 'story-recall',
 'word-color-stroop',
 'breath-sounds',
 

In [73]:
pk_train_c['participant_id'].unique()

<ArrowStringArray>
['110043', '039709', '844158', '024013', '193039', '525316', '891905',
 '403436', '234665', '237791', '982368', '579465', '852129', '267027',
 '686795', '672685', '401731', '505795', '424233', '943245', '881394',
 '090051', '524544', '684449', '371251', '260036', '124647', '420248',
 '394760', '192813', '478433', '024125', '629473', '120723', '075587',
 '108713', '145492', '962050', '748240', '174502', '141858', '427735',
 '312782', '695754', '113118', '958051', '748807', '224730', '959779',
 '013776', '316915', '320532', '833079', '265997', '641284', '968296']
Length: 56, dtype: str

## 4. Handle remaining NaN values
For rows that survived the column drop:
- **Numeric columns**: fill with median (robust to outliers) or leave as-is depending on downstream model.
- **Categorical columns**: fill with `"Unknown"` or most frequent value.
- **Audio features** (`mfcc`, `mel_spectrogram`): should have 0% NaN; flag any issues.

In [74]:
# ══════════════════════════════════════════════════════════════════
#  5. Remaining NaN report + imputation strategy
# ══════════════════════════════════════════════════════════════════

def nan_report(df, name):
    """Show remaining NaN per column after cleaning."""
    n_rows = len(df)
    nan_cols = []
    for col in df.columns:
        n_na = df[col].isna().sum()
        if n_na > 0:
            pct = 100 * n_na / n_rows
            nan_cols.append({"column": col, "NaN": n_na, "NaN%": round(pct, 1),
                             "dtype": str(df[col].dtype)})

    if nan_cols:
        nan_df = pd.DataFrame(nan_cols).sort_values("NaN%", ascending=False)
        print(f"\n{'═' * 70}")
        print(f"  {name} — {len(nan_cols)} columns still have NaN")
        print(f"{'═' * 70}")
        display(nan_df)
    else:
        print(f"\n  ✔  {name} — No NaN values remaining!")

    return pd.DataFrame(nan_cols) if nan_cols else pd.DataFrame()

train_nans = nan_report(pk_train_c, "pk_train_c")


══════════════════════════════════════════════════════════════════════
  pk_train_c — 43 columns still have NaN
══════════════════════════════════════════════════════════════════════


,column,NaN,NaN%,dtype
8,household_income_usa,1212,64.6,float64
19,marital_status___2,950,50.6,float64
29,self_reported_huntingtons,933,49.7,float64
9,housing_status,689,36.7,float64
20,race___5,491,26.2,float64
18,employ_status___7,377,20.1,float64
17,demographics_completed_by___1,374,19.9,float64
5,grandparent,373,19.9,float64
12,other_live_with,373,19.9,float64
13,parent,373,19.9,float64


In [75]:
# ══════════════════════════════════════════════════════════════════
#  5b. Impute remaining NaN values
# ══════════════════════════════════════════════════════════════════

def impute_nans(df, name):
    """
    - Numeric columns: fill NaN with column median
    - Object/string columns: fill NaN with 'Unknown'
    - Audio columns (mfcc, mel_spectrogram): leave as-is (should be 0% NaN)
    """
    df = df.copy()
    skip_cols = {"participant_id", "session_id", "task_name", "mfcc", "mel_spectrogram"}
    filled = {"numeric": 0, "categorical": 0}

    for col in df.columns:
        if col in skip_cols:
            continue
        n_na = df[col].isna().sum()
        if n_na == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            filled["numeric"] += 1
        else:
            df[col] = df[col].fillna("Unknown")
            filled["categorical"] += 1

    remaining_na = df.isna().sum().sum()
    print(f"  [{name}] Imputed: {filled['numeric']} numeric (median), "
          f"{filled['categorical']} categorical ('Unknown')")
    print(f"  [{name}] Remaining NaN: {remaining_na}")
    return df

pk_train_c = impute_nans(pk_train_c, "pk_train_c")

  [pk_train_c] Imputed: 43 numeric (median), 0 categorical ('Unknown')
  [pk_train_c] Remaining NaN: 0


## 6. Final summary & save cleaned tables

In [76]:
# ══════════════════════════════════════════════════════════════════
#  6. Final summary & save
# ══════════════════════════════════════════════════════════════════

def final_summary(df, name):
    n_pids = df["participant_id"].nunique()
    n_sess = df["session_id"].nunique()
    n_tasks = df["task_name"].nunique() if "task_name" in df.columns else 0
    n_na_total = df.isna().sum().sum()

    print(f"{'═' * 70}")
    print(f"  {name}")
    print(f"{'─' * 70}")
    print(f"  Shape:        {df.shape}")
    print(f"  Participants: {n_pids}")
    print(f"  Sessions:     {n_sess}")
    print(f"  Tasks:        {n_tasks}")
    print(f"  Total NaN:    {n_na_total}")
    print(f"  Dtypes:       {df.dtypes.value_counts().to_dict()}")
    print(f"{'═' * 70}\n")

final_summary(pk_train_c, "pk_train_c (non-longitudinal, cleaned)")

# ── Save cleaned table ───────────────────────────────────────────
SAVE_DIR = Path("/home/srl/B2AI/LLM/data")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

clean_path = SAVE_DIR / "cleaned_non_longitudinal_parkinson.parquet"
pk_train_c.to_parquet(clean_path, index=False)

print(f"✅  Saved cleaned table to {SAVE_DIR}/")
print(f"  {clean_path.name:55s}  {pk_train_c.shape}")

══════════════════════════════════════════════════════════════════════
  pk_train_c (non-longitudinal, cleaned)
──────────────────────────────────────────────────────────────────────
  Shape:        (1876, 73)
  Participants: 56
  Sessions:     56
  Tasks:        120
  Total NaN:    0
  Dtypes:       {dtype('float64'): 46, dtype('int64'): 18, Int8Dtype(): 4, <StringDtype(na_value=nan)>: 3, dtype('O'): 2}
══════════════════════════════════════════════════════════════════════

✅  Saved cleaned table to /home/srl/B2AI/LLM/data/
  cleaned_non_longitudinal_parkinson.parquet               (1876, 73)
